# Part 2: Check the estimates

Replay `traces/azure_conversation.csv` in Vidur (vLLM, one A100-80GB), compare it with your Part 1 predictions, then look at what Part 1 could not see: the time between output tokens. Run from the repo root:

```bash
QPS=5.53 bash scripts/run_default.sh traces/azure_conversation.csv
```

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "traces").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from analysis.replay import load_run
from analysis.workload import kv_in_use, load_trace, predict_queue_wait

WAIT_TARGET_S = 2.0    # P99 wait under 2 s (Part 1)
TBT_TARGET_S = 0.100   # P99 time between tokens under 100 ms
BIN_S = 30.0

trace = load_trace(ROOT / "traces" / "azure_conversation.csv")
predicted_wait = predict_queue_wait(trace)
predicted_kv = kv_in_use(trace, predicted_wait, BIN_S)

RUN = ROOT / "simulator_output" / "REPLACE_TIMESTAMP"   # your vLLM run
run = load_run(RUN, trace)
req, kv = run["requests"], run["kv"]

## Predicted vs measured

In [ ]:
t = req.arrived_at.to_numpy()

def longest_late_stretch(wait, merge_gap_s=60.0):
    """Longest time range where requests wait over the target (gaps under a minute merged)."""
    late = t[wait > WAIT_TARGET_S]
    if len(late) == 0:
        return "never"
    breaks = np.flatnonzero(np.diff(late) > merge_gap_s)
    starts, ends = np.r_[late[0], late[breaks + 1]], np.r_[late[breaks], late[-1]]
    i = np.argmax(ends - starts)
    return f"{starts[i]:.0f}-{ends[i]:.0f} s"

print(f"{'':28s}{'predicted':>14s}{'Vidur':>14s}")
print(f"{'P99 wait (s)':28s}{np.quantile(predicted_wait, .99):14.1f}{req.wait.quantile(.99):14.1f}")
print(f"{'longest stretch above 2 s':28s}{longest_late_stretch(predicted_wait):>14s}{longest_late_stretch(req.wait.to_numpy()):>14s}")
print(f"{'requests waiting over 2 s':28s}{np.mean(predicted_wait > WAIT_TARGET_S):14.0%}{np.mean(req.wait > WAIT_TARGET_S):14.0%}")
print(f"{'peak KV cache (GB)':28s}{predicted_kv.kv_gb.max():14.1f}{kv.kv_gb.max():14.1f}")

fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax0.plot(t, req.wait, label="Vidur", linewidth=0.8)
ax0.plot(t, predicted_wait, label="predicted", linewidth=0.8)
ax0.axhline(WAIT_TARGET_S, color="k", linestyle="--", linewidth=0.8, label="2 s target")
ax0.set_ylabel("wait (s)")
ax0.legend(frameon=False)
ax1.plot(kv.time, kv.kv_gb, label="Vidur", linewidth=0.8)
ax1.plot(predicted_kv.bin_start, predicted_kv.kv_gb, label="predicted", linewidth=0.8)
ax1.set_ylabel("KV cache in use (GB)")
ax1.set_xlabel("trace time (s)")
ax1.legend(frameon=False)
fig.tight_layout()
plt.show()

## Time between tokens

In [ ]:
tbt = run["tbt"].set_index("percentile").seconds
print(f"time between tokens: P50 {tbt[50]*1e3:.0f} ms   P90 {tbt[90]*1e3:.0f} ms   P99 {tbt[99]*1e3:.0f} ms   longest {tbt[100]:.1f} s")

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(tbt.to_numpy(), tbt.index / 100)
ax.axvline(TBT_TARGET_S, color="k", linestyle="--", linewidth=0.8, label="100 ms target")
ax.set_xscale("log")
ax.set_xlabel("time between tokens (s)")
ax.set_ylabel("fraction of tokens")
ax.legend(frameon=False)
plt.show()

## Questions

**Q1.** Compare your Part 1 predictions with Vidur: P99 wait, the time range above the 2 s target, and peak KV-cache memory. Which held and which did not? For any that missed, say what the Part 1 model left out.

**Your answer:**

**Q2.** The target for time between tokens is **P99 under 100 ms**. Does vLLM meet it? Report the P50, P99, and longest time between tokens. What does the longest one mean for a user reading the answer?

**Your answer:**

**Q3.** Why do running requests get no new tokens for whole seconds during the surge? Read `_get_next_batch` in `vidur/scheduler/replica_scheduler/vllm_replica_scheduler.py`: when new requests are waiting, what goes into the next iteration, and what has to wait?

**Your answer:**

**Q4.** Part 3 fixes this by capping each iteration at a token budget C: every running request gets its one token, and waiting prompts fill the rest of the budget in chunks. One pass with C tokens takes about **6 ms + C × 78 µs** (Vidur's measured prefill time). Predict the P99 time between tokens for C = 256, 512, and 2048. Which meet the 100 ms target? A smaller C is not free: what happens to the fixed 6 ms per pass, and so to throughput and the queue, when the same prompts take more passes?

| C | Predicted P99 time between tokens |
|---|---|
| 256 | |
| 512 | |
| 2048 | |

**Your answer:**